In [2]:
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import librosa
import os
import random
import shutil
import webrtcvad

import soundfile as sf

In [3]:
# load the trained model
model = YOLO('runs/detect/train/weights/best.pt')

In [4]:
#preprocess .wav file
Recordingfolder = './Recording/'

vad = webrtcvad.Vad()
vad.set_mode(2)  # 0-3 (0 = less aggressive, 3 = most aggressive (cut the background))
frame_duration = 30  # milliseconds

for filename in os.listdir(Recordingfolder):
    file_path = os.path.join(Recordingfolder, filename)  # Construct the full file path
    filename = filename[:-4]
    y, sr = librosa.load(file_path, sr=16000)  # Load the audio file
    y_int16 = (y * 32768).astype(np.int16).tobytes()

    # Divide audio into frames each of 30ms
    frame_size = int(sr * frame_duration / 1000) * 2  # *2 because it is 16-bit (2 bytes)
    frames = [y_int16[i:i+frame_size] for i in range(0, len(y_int16), frame_size)]

    # Ensure frames are valid for webrtcvad
    valid_frames = []
    for i in range(0, len(y_int16), frame_size):
        frame = y_int16[i:i + frame_size]
        if len(frame) == frame_size:  # Skip incomplete frames
            valid_frames.append(frame)

    # Check which frames contain speech
    voiced_frames = [frame for frame in valid_frames if vad.is_speech(frame, sr)]
            
    # Combine voiced frames into a single byte string
    speech_pcm = b''.join(voiced_frames)
    speech_np = np.frombuffer(speech_pcm, dtype=np.int16).astype(np.float32) / 32768

    # Compute mel-spectrogram
    mel_spec = librosa.feature.melspectrogram(y=speech_np, sr=sr, n_mels=128)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)  # Convert to decibel scale

    # Plot the mel-spectrogram
    plt.figure(figsize=(10, 6))
    plt.axis('off')  # Turn off axis
    librosa.display.specshow(mel_spec_db, sr=sr, x_axis='time', y_axis='mel')

    # Save the mel-spectrogram plot as an image
    output_dir = "./ProcessedRecording"
    os.makedirs(output_dir, exist_ok=True)  # Ensure the output directory exists
    plt.savefig(f"{output_dir}/mel_spec_image_{filename}.png", transparent=True)
    plt.close()  # Close the plot to prevent display

In [18]:
# Predict the validation dataset
results = model.predict(source="./ProcessedRecording")

for result in results:
    if len(result.boxes.cls) > 0:
        predicted_class = result.boxes.cls[0].item()
        resultName = result.path.replace("c:\\Users\\riwki\\OneDrive\\Desktop\\UTS\\Neural Network and Fuzzy Logic\\Project\\demonstration\\ProcessedRecording\\", "")
        print(f"{resultName} is predicted as {model.names[int(predicted_class)]} with confidence {result.boxes.conf[0].item()}")



image 1/1 c:\Users\riwki\OneDrive\Desktop\UTS\Neural Network and Fuzzy Logic\Project\demonstration\ProcessedRecording\mel_spec_image_ClaudiaSpeak.png: 192x320 1 Spanish, 64.3ms
Speed: 2.1ms preprocess, 64.3ms inference, 2.2ms postprocess per image at shape (1, 3, 192, 320)
mel_spec_image_ClaudiaSpeak.png is predicted as Spanish with confidence 0.9563636779785156
